<a href="https://colab.research.google.com/github/ErickJester/expo-escom/blob/main/00_conteo_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛠️ Herramientas de Dataset — ExpoEscom
Menú con utilidades sobre una carpeta de Drive (incluye "Compartido conmigo").

**Opciones del menú:**
1. **Contabilizar** — cuenta las imágenes `.jpg` de una carpeta (recursivo)
   y muestra cuántas faltan para llegar a 100k.
2. **Aplanar** — saca las imágenes de una subcarpeta mal anidada y las sube
   a la carpeta superior (las deja donde deberían estar).
3. **Completar a 100k** — cuenta la clase y augmenta **solo el déficit**
   (ej. pokemon 92k → genera 8k) hasta dejar exactamente 100k.

**⚠️ La opción 3 necesita un acceso directo en tu "Mi unidad":**
como `FOLDER_ID` es "Compartido conmigo" no se monta como ruta. Crea **una sola
vez** un acceso directo a esa carpeta en tu *Mi unidad* (en Drive: clic derecho →
Organizar → Añadir acceso directo → Mi unidad) y pon su nombre en
`NOMBRE_ACCESO_DIRECTO` (cell-2).

**Cómo usar:** corre todas las celdas de setup (de arriba) una vez, luego
ejecuta la última celda (Menú) cuantas veces quieras.

In [ ]:
VERSION = '4.0.0'

print('═' * 50)
print('🛠️  Herramientas de Dataset — ExpoEscom')
print(f'v{VERSION}')
print('═' * 50)

!pip install tqdm -q

import logging
logging.getLogger('googleapiclient.http').setLevel(logging.ERROR)

from google.colab import auth
from googleapiclient.discovery import build
from tqdm.notebook import tqdm

auth.authenticate_user()
service = build('drive', 'v3')
print('✅ Autenticado con Drive API')

In [ ]:
# ════ ÚNICA CONFIGURACIÓN NECESARIA ══════════════════════════

FOLDER_ID = '1YwNZMW67NYGMb_g-74kG5gb2u_m_3c1J'

EXTENSIONES_IMAGEN = {'.jpg'}
TARGET_IMAGENES    = 100_000
print(f'Folder ID : {FOLDER_ID}')
print(f'Extensión : solo .jpg  |  Meta: {TARGET_IMAGENES:,} imágenes')

# ════ CONFIG OPCIÓN [3] — AUGMENTACIÓN (completar a 100k) ═════
# FOLDER_ID es "Compartido conmigo" → NO se monta como ruta directa.
# Crea UNA SOLA VEZ un acceso directo a esa carpeta en tu "Mi unidad":
#   en Drive → clic derecho en la carpeta → Organizar → Añadir acceso
#   directo → Mi unidad. Pon abajo el NOMBRE del acceso directo.
NOMBRE_ACCESO_DIRECTO = 'dataset'                       # ← ajusta si lo nombraste distinto
MOUNT_ROOT   = f'/content/drive/MyDrive/{NOMBRE_ACCESO_DIRECTO}'

MAX_AUG      = 30_000   # tope de seguridad: no augmentar más que esto por clase
JPEG_QUALITY = 95
SEED         = 42
NUM_WORKERS  = 4        # hilos paralelos (4 es óptimo en Colab)
print(f'Acceso directo (opción 3): {MOUNT_ROOT}')

In [ ]:
# ════ HELPERS DE DRIVE API ════════════════════════════════════
from pathlib import Path

_ARGS = dict(supportsAllDrives=True, includeItemsFromAllDrives=True,
             pageSize=1000)
_FOLDER_MIME = 'application/vnd.google-apps.folder'


def _es_imagen(f):
    # Solo cuenta archivos con extensión .jpg (sin importar el mimeType)
    return Path(f['name']).suffix.lower() in EXTENSIONES_IMAGEN


def listar_contenido(parent_id):
    """Lee el nivel directo de parent_id.
    Devuelve (subcarpetas, n_imagenes_sueltas):
      subcarpetas = lista de dicts {id, name} ordenada por nombre.
      n_imagenes_sueltas = imágenes que cuelgan directo del nivel."""
    subcarpetas, sueltas, token = [], 0, None
    while True:
        resp = service.files().list(
            q=f"'{parent_id}' in parents and trashed=false",
            fields='nextPageToken, files(id,name,mimeType)',
            pageToken=token, **_ARGS).execute()
        for f in resp.get('files', []):
            if f['mimeType'] == _FOLDER_MIME:
                subcarpetas.append({'id': f['id'], 'name': f['name']})
            elif _es_imagen(f):
                sueltas += 1
        token = resp.get('nextPageToken')
        if not token:
            break
    subcarpetas.sort(key=lambda c: c['name'].lower())
    return subcarpetas, sueltas


def contar_imagenes(folder_id):
    """Cuenta recursivamente los archivos .jpg bajo folder_id.
    Muestra barra de progreso en tiempo real (total desconocido al inicio)."""
    total, stack = 0, [folder_id]
    with tqdm(desc='Contando .jpg', unit=' jpg', dynamic_ncols=True) as pbar:
        while stack:
            fid, token = stack.pop(), None
            while True:
                resp = service.files().list(
                    q=f"'{fid}' in parents and trashed=false",
                    fields='nextPageToken, files(id,name,mimeType)',
                    pageToken=token, **_ARGS).execute()
                for f in resp.get('files', []):
                    if f['mimeType'] == _FOLDER_MIME:
                        stack.append(f['id'])
                    elif _es_imagen(f):
                        total += 1
                        pbar.update(1)
                token = resp.get('nextPageToken')
                if not token:
                    break
    return total


def mover_contenido(origen_id, destino_id):
    """Mueve TODOS los hijos directos de origen_id hacia destino_id.
    En Drive 'mover' es solo cambiar el parent: no re-sube nada.

    Manejo de runs parciales anteriores:
      Si un archivo ya tiene destino_id como parent (la operación fue
      interrumpida a medias), la API devuelve 403 cannotAddParent.
      En ese caso solo eliminamos el parent viejo (origen_id) para
      completar el movimiento sin fallar.

    Devuelve conteos {imagenes, carpetas, otros}."""
    # 1) recolectar todo primero (mover muta el parent, no paginar a la vez)
    print('  Listando archivos a mover…')
    items, token = [], None
    while True:
        resp = service.files().list(
            q=f"'{origen_id}' in parents and trashed=false",
            fields='nextPageToken, files(id,name,mimeType)',
            pageToken=token, **_ARGS).execute()
        items.extend(resp.get('files', []))
        token = resp.get('nextPageToken')
        if not token:
            break

    def _contar(f):
        if f['mimeType'] == _FOLDER_MIME:
            movidos['carpetas'] += 1
        elif _es_imagen(f):
            movidos['imagenes'] += 1
        else:
            movidos['otros'] += 1

    # 2) mover uno por uno con barra de progreso
    movidos = {'imagenes': 0, 'carpetas': 0, 'otros': 0}
    with tqdm(total=len(items), desc='Moviendo', unit=' archivo',
              dynamic_ncols=True) as pbar:
        for f in items:
            try:
                service.files().update(
                    fileId=f['id'], addParents=destino_id,
                    removeParents=origen_id, fields='id',
                    supportsAllDrives=True).execute()
                _contar(f)
            except Exception as e:
                err = str(e)
                if 'cannotAddParent' in err or 'Increasing the number of parents' in err:
                    # El archivo ya tiene destino_id como parent (run parcial anterior).
                    # Solo eliminamos el parent viejo para completar el movimiento.
                    try:
                        service.files().update(
                            fileId=f['id'], removeParents=origen_id,
                            fields='id', supportsAllDrives=True).execute()
                        _contar(f)
                    except Exception as e2:
                        print(f'   ⚠️ No se pudo reparar {f["name"]}: {e2}')
                else:
                    print(f'   ⚠️ No se pudo mover {f["name"]}: {e}')
            pbar.update(1)
    return movidos


def construir_indice(parent_id, incluir_raiz=True):
    """Arma un índice numerado de las subcarpetas de parent_id.
    Si incluir_raiz: [0] representa la carpeta completa."""
    raiz = service.files().get(
        fileId=parent_id, fields='name', supportsAllDrives=True).execute()
    subs, sueltas = listar_contenido(parent_id)
    indice = {}
    if incluir_raiz:
        indice[0] = {'name': f'(TODA la carpeta «{raiz["name"]}»)',
                     'id': parent_id}
    for i, c in enumerate(subs, 1):
        indice[i] = c
    return raiz['name'], indice, sueltas


def mostrar_indice(indice, titulo='ÍNDICE DE CARPETAS'):
    print('═' * 50)
    print(titulo)
    print('═' * 50)
    for num, c in indice.items():
        print(f'  [{num:>2}]  {c["name"]}')
    print('═' * 50)


def pedir_opcion(indice, pregunta):
    """Pide un número válido del índice; reintenta hasta que sea correcto."""
    while True:
        sel = input(f'\n{pregunta} (número): ').strip()
        if sel.isdigit() and int(sel) in indice:
            return int(sel)
        print('   ⚠️  Número inválido, intenta de nuevo.')


print('✅ Helpers listos')

In [ ]:
# ════ PIPELINE DE AUGMENTACIÓN (opción 3) ════════════════════
# Portado TAL CUAL del notebook 09_augmentacion_100k. Solo se
# parametriza por clase/ruta; la lógica de transforms, hilos y
# guardado es idéntica (así funciona bien y no se toca).
import os, re, random, shutil, time, subprocess, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import torchvision
import torchvision.transforms as T
from PIL import Image

_TV = tuple(int(x) for x in torchvision.__version__.split('.')[:2])
VALID_EXT = {'.jpg'}   # dataset 100% .jpg (consistente con la opción [1])


def _listar_drive(path, retries=3):
    """Lista .jpg con shell `find` (resiste FUSE mejor que os.scandir)."""
    for intento in range(retries):
        try:
            result = subprocess.run(
                f'find "{path}" -maxdepth 1 -type f -iname "*.jpg"',
                shell=True, capture_output=True, text=True, timeout=180)
            if result.returncode == 0:
                return [l.strip() for l in result.stdout.splitlines() if l.strip()]
            raise OSError(result.stderr.strip())
        except Exception as e:
            print(f'  ⚠️  Intento {intento+1}/{retries}: {e}')
            if intento < retries - 1:
                from google.colab import drive
                print('  🔄 Re-montando Drive...')
                drive.mount('/content/drive', force_remount=True)
    raise RuntimeError('❌ No se pudo listar la carpeta tras varios intentos.')


def _nuevo_transform():
    """Pipeline de augmentación (un compose por hilo, thread-local)."""
    try:
        _rot = (T.RandomRotation(degrees=12, fill=(210, 210, 210))
                if _TV >= (0, 9) else T.RandomRotation(degrees=12))
    except TypeError:
        _rot = T.RandomRotation(degrees=12)
    return T.Compose([
        T.RandomHorizontalFlip(p=0.5),
        _rot,
        T.RandomResizedCrop(224, scale=(0.88, 1.0), ratio=(0.95, 1.05)),
        T.ColorJitter(brightness=0.20, contrast=0.15, saturation=0.08, hue=0.03),
    ])


def augmentar_clase(clase, class_path):
    """Completa `clase` hasta TARGET_IMAGENES augmentando SOLO el déficit.
    class_path = ruta MONTADA de la carpeta de la clase."""
    class_path = str(class_path)

    # ── Diagnóstico: contar y calcular déficit ───────────────
    print('📋 Listando .jpg (puede tardar ~20s con 70K+ imgs)...')
    source_paths = _listar_drive(class_path)
    print(f'   {len(source_paths):,} .jpg encontrados')

    pat_num = re.compile(r'(\d+)$')
    indices = [int(m.group(1)) for p in source_paths
               if (m := pat_num.search(Path(p).stem))]

    count_existing = len(source_paths)
    max_index      = max(indices) if indices else 0
    next_index     = max_index + 1
    count_needed   = max(0, TARGET_IMAGENES - count_existing)

    SEP = '─' * 55
    print(SEP)
    print(f'  Clase               : {clase}')
    print(f'  .jpg en Drive       : {count_existing:>10,}')
    print(f'  Índice máximo       : {max_index:>10,}  (próximo: {next_index:,})')
    print(f'  TARGET              : {TARGET_IMAGENES:>10,}')
    print(f'  A generar           : {count_needed:>10,}')
    print(f'  Sin número en nombre: {count_existing - len(indices):>9,}  (se usan como fuente)')
    print(SEP)

    # ── Guardián ─────────────────────────────────────────────
    if count_needed == 0:
        print(f'🎉 «{clase}» ya completa — nada que hacer.')
        return
    if count_needed > MAX_AUG:
        print(f'🚫 DETENIDO — necesitas {count_needed:,} augmentaciones')
        print(f'   pero el límite es {MAX_AUG:,} (MAX_AUG en config).')
        print(f'   Con solo {count_existing:,} fuentes, augmentar tanto')
        print(f'   dañaría el dataset. Consigue más imágenes reales primero.')
        return

    ratio = count_needed / count_existing if count_existing else 0
    print(f'✅ {count_needed:,} ≤ {MAX_AUG:,} — dentro del límite seguro.')
    print(f'   Ratio: 1 original genera ~{ratio:.2f} copias en promedio.')

    # ── Generar (paralelo, en local) ─────────────────────────
    LOCAL_OUT = f'/content/aug_work/{clase}'
    os.makedirs(LOCAL_OUT, exist_ok=True)

    _thread_local = threading.local()
    def get_transform():
        if not hasattr(_thread_local, 't'):
            _thread_local.t = _nuevo_transform()
        return _thread_local.t

    random.seed(SEED)
    shuffled = source_paths.copy()
    random.shuffle(shuffled)

    jobs = []
    for i in range(count_needed):
        src = shuffled[i % len(shuffled)]
        dst = os.path.join(LOCAL_OUT, f'{clase}_{next_index + i:06d}.jpg')
        jobs.append((src, dst))
    first_new, last_new = next_index, next_index + count_needed - 1

    def process_one(args):
        src, dst = args
        try:
            img = Image.open(src).convert('RGB')
            get_transform()(img).save(dst, 'JPEG', quality=JPEG_QUALITY)
            return True
        except Exception:
            return False

    print(f'🚀 Augmentando con {NUM_WORKERS} hilos...')
    t_start = time.time()
    generated = errors = 0
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        with tqdm(total=count_needed, unit='img', desc=f'Aug {clase}',
                  dynamic_ncols=True) as pbar:
            for ok in executor.map(process_one, jobs):
                generated += ok
                errors += (not ok)
                pbar.update(1)
    elapsed = time.time() - t_start
    rate = generated / elapsed if elapsed > 0 else 0
    print(f'✅ {generated:,} imgs en {elapsed/60:.1f} min  ({rate:.0f} img/s)')
    if errors:
        print(f'⚠️  {errors} archivos fallaron (se ignoraron)')

    # ── Subir solo los nuevos a Drive ────────────────────────
    pat_new = re.compile(rf'^{re.escape(clase)}_?(\d+)\.jpg$', re.IGNORECASE)
    to_upload = sorted(
        f for f in os.listdir(LOCAL_OUT)
        if (m := pat_new.match(f)) and first_new <= int(m.group(1)) <= last_new)
    print(f'⏫ Subiendo {len(to_upload):,} archivos a Drive...')
    t_up = time.time()
    with tqdm(total=len(to_upload), unit='img', desc='Drive',
              dynamic_ncols=True) as pbar:
        for fname in to_upload:
            shutil.copy2(os.path.join(LOCAL_OUT, fname),
                         os.path.join(class_path, fname))
            pbar.update(1)
    print(f'✅ Subida OK en {(time.time()-t_up)/60:.1f} min → {class_path}')

    # ── Resumen final ────────────────────────────────────────
    final = sum(1 for f in os.scandir(class_path)
                if Path(f.name).suffix.lower() in VALID_EXT)
    print('=' * 55)
    print(f'  RESUMEN  —  {clase}')
    print('=' * 55)
    print(f'  Originales   : {count_existing:>10,}')
    print(f'  Generadas    : {generated:>10,}')
    print(f'  Total Drive  : {final:>10,} / {TARGET_IMAGENES:,}')
    if final >= TARGET_IMAGENES:
        print('  🎉 COMPLETA — lista para entrenamiento')
    else:
        print(f'  ⚠️  Faltan {TARGET_IMAGENES-final:,} — re-ejecuta')
    print('=' * 55)


print('✅ Pipeline de augmentación listo')

In [ ]:
# ════ OPCIONES DEL MENÚ ═══════════════════════════════════════

def opcion_contar():
    """[1] Contabilizar imágenes .jpg de una carpeta."""
    nombre, indice, sueltas = construir_indice(FOLDER_ID, incluir_raiz=True)
    print(f'\n📂 Carpeta raíz : {nombre}')
    mostrar_indice(indice)
    if sueltas:
        print(f'⚠️  {sueltas:,} .jpg sueltos en la raíz '
              f'(la opción [0] los incluye).')

    sel = pedir_opcion(indice, '¿Qué carpeta quieres contabilizar?')
    elegida = indice[sel]
    print(f'\n⏳ Contando .jpg en: {elegida["name"]}…')
    n = contar_imagenes(elegida['id'])

    faltan = max(0, TARGET_IMAGENES - n)
    pct    = n / TARGET_IMAGENES * 100

    print('\n' + '═' * 50)
    print(f'  Carpeta  : {elegida["name"]}')
    print(f'  .jpg     : {n:,}  ({pct:.1f}% de {TARGET_IMAGENES:,})')
    if faltan:
        print(f'  Faltan   : {faltan:,}')
    else:
        print(f'  ✅ Meta alcanzada ({TARGET_IMAGENES:,})')
    print('═' * 50)


def opcion_aplanar():
    """[2] Subir el contenido de una subcarpeta mal anidada a su carpeta padre."""
    # 1) elegir la carpeta donde están las imágenes mal anidadas
    nombre, indice, _ = construir_indice(FOLDER_ID, incluir_raiz=True)
    print(f'\n📂 Carpeta raíz : {nombre}')
    mostrar_indice(indice)
    sel = pedir_opcion(indice, '¿En qué carpeta están las imágenes mal anidadas?')
    destino = indice[sel]

    # 2) buscar subcarpetas dentro de la carpeta elegida
    subs, _ = listar_contenido(destino['id'])
    if not subs:
        print(f'\n✅ «{destino["name"]}» no tiene subcarpetas. Nada que aplanar.')
        return

    subindice = {i: c for i, c in enumerate(subs, 1)}
    mostrar_indice(subindice,
                   titulo=f'SUBCARPETAS DENTRO DE «{destino["name"]}»')
    sub_sel = pedir_opcion(
        subindice, '¿Qué subcarpeta quieres vaciar (subir su contenido)?')
    subcarpeta = subindice[sub_sel]

    # 3) confirmar (operación que modifica tu Drive)
    print('\nSe moverá TODO el contenido de:')
    print(f'   «{subcarpeta["name"]}»   →   «{destino["name"]}»')
    if input('¿Confirmas? (si/no): ').strip().lower() not in ('si', 's', 'sí'):
        print('Cancelado. No se movió nada.')
        return

    # 4) mover
    print('\n⏳ Moviendo…')
    movidos = mover_contenido(subcarpeta['id'], destino['id'])
    total = sum(movidos.values())
    print('\n' + '═' * 50)
    print(f'  Movidos a «{destino["name"]}» : {total:,} elementos')
    print(f'    · .jpg     : {movidos["imagenes"]:,}')
    if movidos['carpetas']:
        print(f'    · carpetas : {movidos["carpetas"]:,}')
    if movidos['otros']:
        print(f'    · otros    : {movidos["otros"]:,}')
    print('═' * 50)

    # 5) opcional: enviar a la papelera la subcarpeta ya vacía
    resp = input(f'\n¿Eliminar la subcarpeta ya vacía «{subcarpeta["name"]}»? '
                 f'(si/no): ').strip().lower()
    if resp in ('si', 's', 'sí'):
        service.files().update(fileId=subcarpeta['id'], body={'trashed': True},
                               supportsAllDrives=True).execute()
        print('   🗑️  Subcarpeta enviada a la papelera.')
    else:
        print('   La subcarpeta vacía se conservó.')


def opcion_augmentar():
    """[3] Completar a 100k: cuenta primero y augmenta SOLO el déficit.
    Usa la ruta MONTADA (acceso directo en MyDrive), igual que el
    notebook de augmentación original."""
    # 1) montar Drive (necesario para leer/escribir vía el acceso directo)
    from google.colab import drive
    drive.mount('/content/drive')

    root = Path(MOUNT_ROOT)
    if not root.is_dir():
        print(f'\n❌ No existe la ruta montada: {root}')
        print('   Crea un acceso directo de la carpeta compartida en tu "Mi unidad"')
        print('   (clic derecho → Organizar → Añadir acceso directo → Mi unidad)')
        print(f'   y verifica NOMBRE_ACCESO_DIRECTO en la config (cell-2).')
        return

    # 2) índice de clases (subcarpetas de la ruta) — mismo UX que [2]
    clases = sorted([d for d in root.iterdir() if d.is_dir()],
                    key=lambda d: d.name.lower())
    if not clases:
        print(f'\n❌ No hay subcarpetas dentro de {root}')
        return

    indice = {i: d for i, d in enumerate(clases, 1)}
    print(f'\n📂 Acceso directo : {root}')
    print('═' * 50)
    print(f'CLASES EN «{root.name}»')
    print('═' * 50)
    for num, d in indice.items():
        print(f'  [{num:>2}]  {d.name}')
    print('═' * 50)

    while True:
        sel = input('\n¿Qué carpeta quieres completar a 100k? (número): ').strip()
        if sel.isdigit() and int(sel) in indice:
            break
        print('   ⚠️  Número inválido, intenta de nuevo.')
    clase_dir = indice[int(sel)]

    # 3) contar + augmentar el déficit (pipeline tal cual)
    print(f'\n⏳ Procesando «{clase_dir.name}»…')
    augmentar_clase(clase_dir.name, clase_dir)


print('✅ Opciones del menú listas')

## ▶️ Menú — ejecuta esta celda

In [ ]:
# ════ MENÚ PRINCIPAL ══════════════════════════════════════════
print('═' * 50)
print('  MENÚ PRINCIPAL')
print('═' * 50)
print('  [1]  Contabilizar imágenes de una carpeta')
print('  [2]  Aplanar: subir el contenido de una subcarpeta')
print('  [3]  Completar a 100k: augmentar el déficit')
print('═' * 50)

_op = input('Elige una opción (1/2/3): ').strip()
if _op == '1':
    opcion_contar()
elif _op == '2':
    opcion_aplanar()
elif _op == '3':
    opcion_augmentar()
else:
    print('Opción inválida. Vuelve a ejecutar esta celda.')